# Step 3 — Model Boolean

**Tujuan praktikum**
1. Mahasiswa memahami model Boolean dan operator AND, OR, NOT.
2. Mahasiswa mampu mengimplementasikan algoritma *merge* (intersection) dua posting list terurut.
3. Mahasiswa mampu mengoptimasi urutan pemrosesan query berdasarkan *document frequency*.
4. Mahasiswa memahami keterbatasan model Boolean (tidak ada peringkat).

In [7]:
# Jalankan sekali jika pustaka belum terpasang:
!pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.6 MB/s eta 0:00:00


In [13]:
import os
import re
import sys
import shutil
from pathlib import Path

FILE_WAJIB = ["stki_utils.py", "korpus.csv", "queries.csv", "qrels.csv"]
WORKDIR = Path.cwd()

def file_hilang():
    return [f for f in FILE_WAJIB if not (WORKDIR / f).exists()]

def rapikan_nama():
    """Colab/browser kadang mengganti nama file ('korpus (1).csv', 'stki_utils_1.py').
    Salin versi terbaru ke nama standar yang dibutuhkan notebook."""
    for nama in FILE_WAJIB:
        stem, suf = os.path.splitext(nama)
        pola = re.compile(rf"^{re.escape(stem)}(?:[ _]?\(?\d+\)?)?{re.escape(suf)}$")
        cocok = [p for p in WORKDIR.iterdir() if p.is_file() and pola.match(p.name)]
        if cocok:
            terbaru = max(cocok, key=lambda p: p.stat().st_mtime)
            if terbaru.name != nama:
                shutil.copyfile(terbaru, WORKDIR / nama)
                print(f"Memakai '{terbaru.name}' sebagai '{nama}'")

rapikan_nama()
if file_hilang():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"File berikut tidak ditemukan di {WORKDIR}: {file_hilang()}")
    print("Silakan upload file berikut:", file_hilang())
    files.upload()
    rapikan_nama()

if file_hilang():
    raise FileNotFoundError(
        f"File masih belum lengkap: {file_hilang()}. Jalankan ulang sel ini dan upload semua file "
        "(stki_utils.py, korpus.csv, queries.csv, qrels.csv)."
    )
print("Semua file tersedia di:", WORKDIR)

Semua file tersedia di: /content


In [14]:
# Impor stki_utils dan arahkan DATA_DIR ke folder tempat CSV di-upload.
# (Di stki_utils.py aslinya DATA_DIR = ../data, yang tidak ada di Colab.)
import os
import re
import sys
import time
import random
from pathlib import Path
import pandas as pd
from IPython.display import display

# Aman walau sel upload di atas belum dijalankan / runtime baru di-restart
if "WORKDIR" not in globals():
    WORKDIR = Path.cwd()
if not (WORKDIR / "stki_utils.py").exists():
    raise FileNotFoundError(
        f"stki_utils.py tidak ada di {WORKDIR}. Jalankan sel 'Upload file' di atas lalu pilih "
        "stki_utils.py, korpus.csv, queries.csv, dan qrels.csv."
    )

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import stki_utils
stki_utils.DATA_DIR = WORKDIR

# Kompatibilitas nama kolom ID dokumen: korpus.csv memakai 'doc_id', sedangkan beberapa versi
# stki_utils.py membaca 'id_doc'. Kedua nama disediakan agar versi stki_utils mana pun bisa berjalan.
if not getattr(stki_utils.load_corpus, "_kompatibel", False):
    _load_corpus_asli = stki_utils.load_corpus
    def _load_corpus_kompatibel():
        df = _load_corpus_asli()
        if "doc_id" in df.columns and "id_doc" not in df.columns:
            df["id_doc"] = df["doc_id"]
        elif "id_doc" in df.columns and "doc_id" not in df.columns:
            df["doc_id"] = df["id_doc"]
        return df
    _load_corpus_kompatibel._kompatibel = True
    stki_utils.load_corpus = _load_corpus_kompatibel

from stki_utils import (corpus_texts, preprocess, build_inverted_index,
                        build_positional_index, boolean_search)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

df_korpus = stki_utils.load_corpus()
queries = stki_utils.load_queries()
qrels = stki_utils.load_qrels()
JUDUL = dict(zip(df_korpus.doc_id, df_korpus.judul))

print(f"Jumlah dokumen : {len(df_korpus)}")
print(f"Jumlah query   : {len(queries)}")
print(f"Jumlah qrels   : {sum(len(v) for v in qrels.values())} pasangan (query, dokumen)")
display(df_korpus[["doc_id", "judul"]].head(5))

Jumlah dokumen : 30
Jumlah query   : 8
Jumlah qrels   : 25 pasangan (query, dokumen)


,doc_id,judul
0,D01,Demam Berdarah Dengue
1,D02,Pencegahan Stunting pada Balita
2,D03,Manfaat Olahraga bagi Jantung
3,D04,Mengenal Diabetes Melitus
4,D05,Pentingnya Imunisasi Dasar


In [15]:
from stki_utils import corpus_texts, preprocess, build_inverted_index

docs = {d: preprocess(t) for d, t in corpus_texts().items()}
index = build_inverted_index(docs)
SEMUA_DOC = sorted(docs)

def postings(term):
    t = preprocess(term, use_stopword=False)
    return sorted(index.get(t[0], {})) if t else []

print("tani   :", postings("tani"))
print("air    :", postings("air"))

tani   : ['D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D23']
air    : ['D01', 'D07', 'D09', 'D10']


## Langkah 1 — Algoritma intersection (AND) dengan dua pointer
Karena posting list terurut, irisan dapat dihitung dalam waktu O(x + y) dengan menggeser dua pointer.

In [16]:
def intersect(p1, p2):
    hasil, i, j, perbandingan = [], 0, 0, 0
    while i < len(p1) and j < len(p2):
        perbandingan += 1
        if p1[i] == p2[j]:
            hasil.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return hasil, perbandingan

hasil, n = intersect(postings("petani"), postings("air"))
print("petani AND air =", hasil, f"({n} perbandingan)")

petani AND air = ['D07', 'D09', 'D10'] (5 perbandingan)


## Langkah 2 — Union (OR) dan NOT

In [17]:
def union(p1, p2):
    hasil, i, j = [], 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            hasil.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            hasil.append(p1[i]); i += 1
        else:
            hasil.append(p2[j]); j += 1
    return hasil + p1[i:] + p2[j:]

def and_not(p1, p2):
    s2 = set(p2)
    return [d for d in p1 if d not in s2]

print("pupuk OR kompos     =", union(postings("pupuk"), postings("kompos")))
print("tani AND NOT hama   =", and_not(postings("tani"), postings("hama")))

pupuk OR kompos     = ['D08', 'D10']
tani AND NOT hama   = ['D08', 'D09', 'D10', 'D12', 'D23']


## Langkah 3 — Optimasi query: proses term dengan df terkecil lebih dulu
Untuk query `t1 AND t2 AND ... AND tn`, hasil antara tidak mungkin lebih besar dari posting list terpendek. Mengurutkan term berdasarkan df menaikkan efisiensi.

In [18]:
def and_query(terms, optimasi=True):
    lists = [postings(t) for t in terms]
    if optimasi:
        lists.sort(key=len)
    hasil, total = lists[0], 0
    for p in lists[1:]:
        hasil, n = intersect(hasil, p)
        total += n
    return hasil, total

query = ["data", "aplikasi", "sensor"]
print("df:", {t: len(postings(t)) for t in query})
print("Tanpa optimasi :", and_query(query, optimasi=False))
print("Dengan optimasi:", and_query(query, optimasi=True))

df: {'data': 6, 'aplikasi': 3, 'sensor': 2}
Tanpa optimasi : (['D10'], 8)
Dengan optimasi: (['D10'], 5)


## Langkah 4 — Parser query Boolean lengkap
`stki_utils.boolean_search` mem-parsing query dengan operator `AND`, `OR`, `NOT` (huruf kapital) dan tanda kurung, dengan prioritas NOT > AND > OR. Pelajari kodenya di `stki_utils.py`.

In [19]:
from stki_utils import boolean_search

contoh_query = [
    "petani AND pupuk",
    "pupuk OR kompos",
    "tani AND NOT hama",
    "(digital OR daring) AND internet",
    "gizi AND (anak OR balita) AND NOT sekolah",
]
for q in contoh_query:
    print(f"{q:45s} -> {boolean_search(q, index, SEMUA_DOC)}")

petani AND pupuk                              -> ['D08', 'D10']
pupuk OR kompos                               -> ['D08', 'D10']
tani AND NOT hama                             -> ['D08', 'D09', 'D10', 'D12', 'D23']
(digital OR daring) AND internet              -> ['D25']
gizi AND (anak OR balita) AND NOT sekolah     -> ['D02']


## Langkah 5 — Keterbatasan model Boolean
Perhatikan dua query berikut. Query AND terlalu ketat (sering kosong), sedangkan query OR terlalu longgar dan semua hasil dianggap sama relevannya karena tidak ada peringkat.

In [20]:
q = "keamanan data pribadi di internet"
terms = preprocess(q)
print("Term query:", terms)
print("AND :", boolean_search(" AND ".join(terms), index, SEMUA_DOC))
print("OR  :", boolean_search(" OR ".join(terms), index, SEMUA_DOC))

Term query: ['aman', 'data', 'pribadi', 'internet']
AND : []
OR  : ['D06', 'D10', 'D14', 'D15', 'D16', 'D18', 'D25', 'D27']


## Tugas Step 3
1. Implementasikan fungsi `or_not(p1, p2)` untuk `t1 OR NOT t2` menggunakan daftar semua dokumen. Mengapa operasi ini mahal pada koleksi besar?
2. Buat 5 query Boolean untuk mencari dokumen topik **ekonomi** tanpa mengambil dokumen topik lain. Laporkan hasil tiap query dan jelaskan strategi Anda.
3. Tambahkan dukungan *phrase query* (dalam tanda kutip) ke parser Boolean dengan memanfaatkan positional index dari Step 2.
4. Diskusikan: mengapa model Boolean masih digunakan pada pencarian dokumen hukum atau paten?

## Tugas 1 — Fungsi `or_not(p1, p2)` untuk `t1 OR NOT t2`
`NOT t2` berarti **semua dokumen dalam koleksi yang tidak mengandung t2**. Karena itu operasi ini membutuhkan daftar seluruh dokumen
(`SEMUA_DOC`)

In [21]:
def komplemen(p, semua_doc):
    """NOT p: semua dokumen yang tidak ada di p. Dua pointer, O(N). Mengembalikan (hasil, langkah)."""
    hasil, j, langkah = [], 0, 0
    for d in semua_doc:
        langkah += 1
        if j < len(p) and p[j] == d:
            j += 1
        else:
            hasil.append(d)
    return hasil, langkah

def or_not(p1, p2, semua_doc):
    """t1 OR NOT t2 = p1 UNION (semua_doc - p2). Mengembalikan (hasil, perkiraan_jumlah_operasi)."""
    not_p2, langkah = komplemen(p2, semua_doc)
    hasil = union(p1, not_p2)
    langkah += len(p1) + len(not_p2)       # biaya merge pada union (batas atas)
    return hasil, langkah

# Contoh pada korpus proyek
hasil, op = or_not(postings("hama"), postings("tani"), SEMUA_DOC)
print("hama OR NOT tani =", hasil)
print(f"Jumlah hasil = {len(hasil)} dari {len(SEMUA_DOC)} dokumen; operasi = {op}")

# Verifikasi terhadap parser Boolean bawaan (harus identik)
assert hasil == boolean_search("hama OR NOT tani", index, SEMUA_DOC)
print("Verifikasi dengan boolean_search: COCOK")

hama OR NOT tani = ['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D11', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D24', 'D25', 'D26', 'D27', 'D28', 'D29', 'D30']
Jumlah hasil = 25 dari 30 dokumen; operasi = 55
Verifikasi dengan boolean_search: COCOK


**Mengapa operasi ini mahal pada koleksi besar?** Simulasi berikut memakai koleksi sintetis dengan ukuran N yang bertambah,
sementara posting list `t1` dan `t2` dibuat tetap kecil (50 dokumen, seperti term yang jarang muncul).
Bandingkan biaya `AND` (bergantung pada panjang posting list) dengan `OR NOT` (bergantung pada N).

In [22]:
random.seed(42)
baris = []
for N in [1_000, 10_000, 100_000, 1_000_000]:
    semua = [f"D{i:07d}" for i in range(N)]            # urutan leksikografis = urutan numerik
    p1 = sorted(random.sample(semua, 50))
    p2 = sorted(random.sample(semua, 50))

    t0 = time.perf_counter()
    hasil_or_not, operasi = or_not(p1, p2, semua)
    ms_or_not = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _, bandingan_and = intersect(p1, p2)
    ms_and = (time.perf_counter() - t0) * 1000

    baris.append({"N dokumen": N, "df(t1)": len(p1), "df(t2)": len(p2),
                  "|hasil OR NOT|": len(hasil_or_not),
                  "operasi OR NOT": operasi, "waktu OR NOT (ms)": round(ms_or_not, 2),
                  "operasi AND": bandingan_and, "waktu AND (ms)": round(ms_and, 4)})

display(pd.DataFrame(baris).set_index("N dokumen"))

,df(t1),df(t2),|hasil OR NOT|,operasi OR NOT,waktu OR NOT (ms),operasi AND,waktu AND (ms)
N dokumen,,,,,,,
1000,50,50,953,2000,0.34,96,0.0259
10000,50,50,9951,20000,3.57,98,0.0288
100000,50,50,99950,200000,39.99,99,0.0346
1000000,50,50,999950,2000000,391.05,96,0.0625


**Penjelasan.**
- `t1 AND t2` hanya menelusuri posting list, biaya **O(df₁ + df₂)**, tidak bergantung pada ukuran koleksi.
- `t1 OR NOT t2` harus membentuk komplemen `NOT t2` yang berukuran **N − df₂**. Untuk term yang jarang muncul, komplemen itu
  hampir sama dengan seluruh koleksi, sehingga biaya waktu dan memori menjadi **O(N)** dan hasil akhirnya pun hampir selalu berukuran ≈ N.
  Pada tabel terlihat operasi `OR NOT` tumbuh linear terhadap N, sedangkan `AND` tetap konstan.
- Hasilnya juga tidak informatif, hampir semua dokumen dikembalikan. Karena itu mesin pencari nyata menolak query yang
  hanya berisi `NOT` atau menulis ulang menjadi bentuk `AND NOT`, atau memakai representasi *bitmap* / komplemen implisit agar tidak mematerialisasi daftar besar.

## Tugas 2 — Lima query Boolean untuk topik **ekonomi**

In [26]:
query_ekonomi = [
    "inflasi OR saham OR kredit OR qris OR kopi OR umkm",       # 1. gabungan kata khas domain (recall tinggi)
    "(kredit OR modal) AND usaha",                               # 2. pembiayaan usaha
    "bank AND NOT digital",                                      # 3. perbankan konvensional
    "(harga OR pasar) AND NOT (petani OR gabah)",                # 4. harga/pasar, buang topik pertanian
    "(investasi OR saham OR bursa) AND risiko",                  # 5. investasi, kunci dengan 'risiko'
]

baris = []
for i, q in enumerate(query_ekonomi, start=1):
    hasil = boolean_search(q, index, SEMUA_DOC)
    benar = set(hasil) & EKONOMI
    bocor = sorted(set(hasil) - EKONOMI)
    baris.append({
        "No": i,
        "Query": q,
        "Hasil": ", ".join(hasil) if hasil else "(kosong)",
        "Jumlah": len(hasil),
        "Precision": round(len(benar) / len(hasil), 3) if hasil else 0.0,
        "Recall": round(len(benar) / len(EKONOMI), 3),
        "Dokumen non-ekonomi": ", ".join(bocor) if bocor else "-",
    })

df_ekonomi = pd.DataFrame(baris).set_index("No")
display(df_ekonomi)

semua_hasil = set()
for q in query_ekonomi:
    semua_hasil |= set(boolean_search(q, index, SEMUA_DOC))
print("Gabungan dokumen yang tertangkap kelima query:", sorted(semua_hasil))
print("Dokumen ekonomi yang belum tertangkap       :", sorted(EKONOMI - semua_hasil) or "-")

,Query,Hasil,Jumlah,Precision,Recall,Dokumen non-ekonomi
No,,,,,,
1,inflasi OR saham OR kredit OR qris OR kopi OR umkm,"D19, D20, D21, D22, D23, D24",6,1.0,1.000,-
2,(kredit OR modal) AND usaha,D22,1,1.0,0.167,-
3,bank AND NOT digital,"D20, D22",2,1.0,0.333,-
4,(harga OR pasar) AND NOT (petani OR gabah),"D19, D20, D24",3,1.0,0.500,-
5,(investasi OR saham OR bursa) AND risiko,D24,1,1.0,0.167,-


Gabungan dokumen yang tertangkap kelima query: ['D19', 'D20', 'D21', 'D22', 'D23', 'D24']
Dokumen ekonomi yang belum tertangkap       : -


- Query 1 (`OR` dari kata khas) menangkap seluruh dokumen ekonomi, setiap term hanya muncul di topik ekonomi.
- Query 2 dan 5 (`AND`) sangat selektif dengan precision tinggi tetapi recall rendah, karena `AND` mensyaratkan semua kondisi terpenuhi.
- Query 3 dan 4 memanfaatkan `NOT` untuk menyingkirkan dokumen topik lain yang memakai kata yang sama (`digital`, `petani`, `gabah`).
- Makin ketat query, makin tinggi precision dan makin rendah recall. Model Boolean tidak punya cara menyeimbangkan keduanya secara otomatis, pengguna harus merancang query secara manual.

## Tugas 3 — Phrase query (dalam tanda kutip) dengan positional index

In [27]:
docs_pos = {d: preprocess(t, use_stopword=False) for d, t in corpus_texts().items()}
pos_index = build_positional_index(docs_pos)

def phrase_postings(phrase, pos_index):
    """Dokumen (terurut) yang memuat frasa persis berurutan."""
    terms = preprocess(phrase, use_stopword=False)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set(pos_index[terms[0]])
    for t in terms[1:]:
        kandidat &= set(pos_index[t])
    hasil = []
    for d in sorted(kandidat):
        posisi_lain = [set(pos_index[t][d]) for t in terms[1:]]
        for p in pos_index[terms[0]][d]:
            if all((p + i) in s for i, s in enumerate(posisi_lain, start=1)):
                hasil.append(d)
                break
    return hasil

print('"mesin pencari"      :', phrase_postings("mesin pencari", pos_index))
print('"pupuk organik"      :', phrase_postings("pupuk organik", pos_index))

"mesin pencari"      : ['D13']
"pupuk organik"      : ['D08']


In [28]:
def boolean_search_phrase(query, index, pos_index, all_docs):
    """
    Query Boolean dengan dukungan phrase query ("...").
    Operator: AND, OR, NOT (huruf kapital), tanda kurung. Prioritas NOT > AND > OR.
    Contoh: '"mesin pencari" OR ("pupuk organik" AND NOT hama)'
    """
    if query.count('"') % 2 != 0:
        raise ValueError("Tanda kutip tidak seimbang")
    all_docs = set(all_docs)
    tokens = re.findall(r'"[^"]*"|\(|\)|[^\s()"]+', query)
    pos = 0

    def peek():
        return tokens[pos] if pos < len(tokens) else None

    def take():
        nonlocal pos
        if pos >= len(tokens):
            raise ValueError("Query berakhir sebelum lengkap (operan hilang)")
        pos += 1
        return tokens[pos - 1]

    def parse_or():
        hasil = parse_and()
        while peek() == "OR":
            take()
            hasil = hasil | parse_and()
        return hasil

    def parse_and():
        hasil = parse_not()
        while peek() == "AND":
            take()
            hasil = hasil & parse_not()
        return hasil

    def parse_not():
        if peek() == "NOT":
            take()
            return all_docs - parse_not()
        if peek() == "(":
            take()
            hasil = parse_or()
            if peek() != ")":
                raise ValueError("Kurung tidak seimbang")
            take()
            return hasil
        tok = take()
        if tok in ("AND", "OR", ")"):
            raise ValueError(f"Operan hilang di dekat '{tok}'")
        if tok.startswith('"'):
            return set(phrase_postings(tok[1:-1], pos_index))
        term = preprocess(tok, use_stopword=False)
        return set(index.get(term[0], {})) if term else set()

    hasil = parse_or()
    if pos != len(tokens):
        raise ValueError(f"Query tidak valid di dekat '{tokens[pos]}'")
    return sorted(hasil)

for q in contoh_query + query_ekonomi:
    assert boolean_search_phrase(q, index, pos_index, SEMUA_DOC) == boolean_search(q, index, SEMUA_DOC), q
print("Query tanpa phrase: hasil identik dengan boolean_search bawaan.")

Query tanpa phrase: hasil identik dengan boolean_search bawaan.


In [29]:
uji_phrase = [
    '"mesin pencari"',
    '"pupuk organik"',
    'harga AND petani',                                  # tanpa phrase: kedua kata ada di dokumen yang sama
    '"harga petani"',                                    # dengan phrase: harus berdampingan -> tidak ada
    '"pembelajaran daring" OR "literasi digital"',
    '"layanan kesehatan" AND NOT hama',
    '"kata dasar" AND (stemming OR tokenisasi)',
]
for q in uji_phrase:
    print(f"{q:50s} -> {boolean_search_phrase(q, index, pos_index, SEMUA_DOC)}")

"mesin pencari"                                    -> ['D13']
"pupuk organik"                                    -> ['D08']
harga AND petani                                   -> ['D12', 'D23']
"harga petani"                                     -> []
"pembelajaran daring" OR "literasi digital"        -> ['D25', 'D27']
"layanan kesehatan" AND NOT hama                   -> ['D06']
"kata dasar" AND (stemming OR tokenisasi)          -> ['D17']


In [30]:
query_salah = ['"harga petani', 'AND pupuk', 'pupuk AND', '(pupuk OR kompos']
for q in query_salah:
    try:
        boolean_search_phrase(q, index, pos_index, SEMUA_DOC)
        print(f"{q!r:25s} -> (tidak error)")
    except ValueError as e:
        print(f"{q!r:25s} -> ValueError: {e}")

'"harga petani'           -> ValueError: Tanda kutip tidak seimbang
'AND pupuk'               -> ValueError: Operan hilang di dekat 'AND'
'pupuk AND'               -> ValueError: Query berakhir sebelum lengkap (operan hilang)
'(pupuk OR kompos'        -> ValueError: Kurung tidak seimbang


## Tugas 4 — Mengapa model Boolean masih digunakan pada pencarian dokumen hukum atau paten?

1. Kebutuhan recall yang hampir total.
Pencari paten dan pengacara harus menemukan semua dokumen yang relevan. Melewatkan satu paten atau satu putusan bisa berakibat paten ditolak atau kasus kalah. Pada model berperingkat, pengguna tidak tahu di peringkat berapa harus berhenti membaca, sedangkann pada Boolean, seluruh himpunan yang cocok bisa diperiksa tuntas.
2. Kontrol dan presisi ekspresi.
Pencari profesional dapat menyusun query kompleks dengan `AND`, `OR`, `NOT`, tanda kurung, *phrase*, operator kedekatan (*proximity*), dan *wildcard* untuk menangkap konsep, sinonim, serta variasi istilah teknis yang eksak (misalnya nomor pasal, kode klasifikasi IPC, nama pihak).
3. Hasil deterministik dan dapat dijelaskan.
Sebuah dokumen muncul karena memenuhi ekspresi logika tertentu, tidak bergantung pada bobot statistik yang buram. Ini penting untuk auditabilitas
4. Reproducibility. Query yang sama pada koleksi yang sama selalu menghasilkan himpunan yang sama dan perubahan statistik koleksi (df, panjang rata-rata dokumen) tidak mengubah hasil. Pada model berperingkat, penambahan dokumen dapat mengubah skor dan urutan.
5. Konsistensi dengan struktur dokumen. Dokumen hukum dan paten bersifat terstruktur (bidang judul, abstrak, klaim, tanggal, kelas), sehingga pembatasan per bidang dan rentang tanggal dengan operator logika sangat natural.
